In [19]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("image-text-to-text", model="Salesforce/blip2-opt-2.7b")

ModuleNotFoundError: No module named 'transformers'

In [6]:
from transformers import pipeline
import re
from typing import Dict, List, Optional

class ResumeImageAnalyzer:
    def __init__(self):
        self.pipe=pipe
        
    def analyze_resume(self, image_path: str) -> Dict[str, str]:
        """
        Analyzes a resume image and extracts key information using different prompts.
        
        Args:
            image_path: Path to the resume image file
            
        Returns:
            Dictionary containing extracted information
        """
        prompts = {
            "contact_info": "What contact information is visible in this resume? Extract email, phone, and location.",
            "education": "List all educational qualifications mentioned in this resume with institutions and years.",
            "work_experience": "Describe the work experience section of this resume including company names, roles, and dates.",
            "skills": "What technical and professional skills are listed in this resume?",
            "certifications": "List any professional certifications or licenses mentioned in this resume.",
            "summary": "Provide a brief professional summary based on this resume."
        }
        
        results = {}
        for key, prompt in prompts.items():
            response = self.pipe(images=image_path, text=prompt)
            results[key] = self._clean_response(response[0])
            
        return results
    
    def extract_specific_detail(self, image_path: str, custom_prompt: str) -> str:
        """
        Extracts specific information using a custom prompt.
        Args:
            image_path: Path to the resume image file
            custom_prompt: Specific prompt to extract targeted information
            
        Returns:
            Extracted information as string
        """
        response = self.pipe(images=image_path, text=custom_prompt)
        return self._clean_response(response[0])
    
    def get_ats_friendly_text(self, image_path: str) -> str:
        """
        Converts resume image to ATS-friendly text format.
        
        Args:
            image_path: Path to the resume image file
            
        Returns:
            ATS-friendly text version of the resume
        """
        prompt = "Convert this resume into plain text format suitable for ATS systems. Include all visible text while maintaining proper section hierarchy."
        response = self.pipe(images=image_path, text=prompt)
        return self._clean_response(response[0])
    
    def _clean_response(self, text: str) -> str:
        """
        Cleans and formats the model's response.
        """
        # Remove multiple spaces and newlines
        text = re.sub(r'\s+', ' ', text).strip()
        # Remove any generated markers or artifacts
        text = re.sub(r'<.*?>', '', text)
        return text

# Example usage and helpful prompts
if __name__ == "__main__":
    analyzer = ResumeImageAnalyzer()
    
    # Example image path
    resume_image = "ocr image.png"
    
    # Get complete analysis
    results = analyzer.analyze_resume(resume_image)
    
    # Get ATS-friendly version
    ats_text = analyzer.get_ats_friendly_text(resume_image)
    
    # Example custom prompts for specific extractions
    custom_prompts = [
        "What programming languages are mentioned in this resume?",
        "Extract all project names and their descriptions from this resume.",
        "List all leadership roles or responsibilities mentioned in this resume.",
        "What achievements or awards are mentioned in this resume?",
        "Extract all dates of employment and calculate total years of experience.",
        "What tools and technologies are mentioned in this resume?",
        "Extract all quantifiable achievements (numbers, percentages, metrics) from this resume.",
        "List all soft skills mentioned in this resume.",
        "Extract information about team size and management experience.",
        "What languages (speaking/writing) are mentioned in this resume?"
    ]
    
    # Extract specific details
    for prompt in custom_prompts:
        detail = analyzer.extract_specific_detail(resume_image, prompt)
        print(f"\nPrompt: {prompt}")
        print(f"Result: {detail}")

Keyword argument `legacy` is not a valid argument for this processor and will be ignored.


TypeError: expected string or bytes-like object, got 'dict'

In [ ]:
# Load model directly
from transformers import AutoProcessor, AutoModelForVisualQuestionAnswering

processor = AutoProcessor.from_pretrained("Salesforce/blip2-opt-2.7b")
model = AutoModelForVisualQuestionAnswering.from_pretrained("Salesforce/blip2-opt-2.7b")

In [3]:
from transformers import AutoProcessor, AutoModelForVisionText2Text
import torch
from PIL import Image

def extract_education(image_path,model):
    # Load the model and processor
    # model = AutoModelForVisionText2Text.from_pretrained("Salesforce/blip2-opt-2.7b")
    processor = AutoProcessor.from_pretrained("Salesforce/blip2-opt-2.7b")
    
    # Load and preprocess the image
    image = Image.open(image_path)
    
    # Prepare the prompt
    prompt = """Please extract the education information from this resume image. 
    Focus on:
    1. School names
    2. Degree types
    3. Majors
    4. Graduation dates
    5. Location
    6. Any awards or leadership positions
    
    Format the response as a structured list."""
    
    # Process image and generate text
    inputs = processor(image, text=prompt, return_tensors="pt")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=500,
            num_beams=5,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.5
        )
    
    # Decode and return the response
    generated_text = processor.decode(outputs[0], skip_special_tokens=True)
    return generated_text

def parse_education_data(extracted_text):
    """Parse the extracted text into a structured format"""
    # Add parsing logic here to convert the text into a structured format
    # This can be customized based on your needs
    return extracted_text

# Usage example
if __name__ == "__main__":
    image_path = "ocr image.png"
    education_info = extract_education(image_path)
    print("Extracted Education Information:")
    print(education_info)

ImportError: cannot import name 'AutoModelForVisionText2Text' from 'transformers' (/opt/anaconda3/envs/AS/lib/python3.11/site-packages/transformers/__init__.py)

In [ ]:
# Different types of prompts for experience extraction

def get_detailed_experience_prompt():
    return """
    Analyze this resume image and extract ALL professional experience information with precise details:
    
    For each position, provide:
    1. Company Name
    2. Job Title
    3. Employment Duration (Start Date - End Date)
    4. Location
    5. Key Responsibilities
    6. Quantifiable Achievements
    7. Team Size and Management Experience
    8. Projects Handled
    9. Technologies or Tools Used
    10. Budget Management Details
    
    Format requirements:
    - Maintain chronological order (most recent first)
    - Preserve all numerical metrics and percentages
    - Include all leadership and management responsibilities
    - Keep original achievement metrics
    
    Structure the response as:
    {Company Name} | {Job Title}
    Duration: {Start Date} - {End Date}
    Location: {Location}
    Key Points:
    - [List all major points with metrics]
    """

def get_concise_experience_prompt():
    return """
    Extract core professional experience from this resume:
    
    Required for each role:
    - Company + Title
    - Dates
    - Location
    - Top 3 achievements with metrics
    
    Format: Bullet points only, most recent first.
    Focus on numerical results and direct impact.
    """

def get_metrics_focused_prompt():
    return """
    From this resume image, extract ONLY the following metrics from all professional experiences:
    
    Find and list all:
    1. Revenue impacts ($ amounts)
    2. Percentage improvements
    3. Team sizes
    4. Project scopes
    5. Cost savings
    6. Time reductions
    7. Growth metrics
    
    Format: Category: Metric (Position + Company)
    """

def get_leadership_experience_prompt():
    return """
    Analyze this resume for leadership and management experience:
    
    Extract:
    1. Team sizes managed
    2. Projects led
    3. Budget responsibility
    4. Strategic initiatives
    5. Mentorship/training programs
    6. Cross-functional leadership
    7. Department/location management
    
    Include all quantifiable metrics related to leadership impact.
    """

def get_technical_experience_prompt():
    return """
    Extract all technical experience details:
    
    Focus areas:
    1. Technical tools/software used
    2. Technologies implemented
    3. Technical projects delivered
    4. System improvements
    5. Technical metrics (performance, efficiency)
    6. Scale of technical impact
    
    Include specific versions/names of tools and technologies.
    """

def get_project_focused_prompt():
    return """
    Extract detailed project experience from this resume:
    
    For each project mentioned:
    1. Project scope
    2. Timeline
    3. Team size
    4. Budget
    5. Technologies used
    6. Measurable outcomes
    7. Role in project
    8. Business impact
    
    Format: Project-by-project breakdown with metrics.
    """

# Example usage with vision model
def extract_resume_info(image_path, prompt_type="detailed"):
    # Initialize your vision model here
    prompts = {
        "detailed": get_detailed_experience_prompt(),
        "concise": get_concise_experience_prompt(),
        "metrics": get_metrics_focused_prompt(),
        "leadership": get_leadership_experience_prompt(),
        "technical": get_technical_experience_prompt(),
        "project": get_project_focused_prompt()
    }
    
    selected_prompt = prompts.get(prompt_type, get_detailed_experience_prompt())
    
    # Add your model inference code here
    # Example:
    # result = model.generate(image=image_path, prompt=selected_prompt)
    # return result
    
    return selected_prompt  # For demonstration

# Example combining multiple prompts for comprehensive extraction
def get_comprehensive_extraction():
    comprehensive_prompt = """
    Perform a detailed analysis of this resume image and extract the following information:

    EXPERIENCE SECTION:
    For each position:
    1. Company: [Company Name]
    2. Title: [Job Title]
    3. Duration: [Start Date] - [End Date]
    4. Location: [City, State/Country]
    
    KEY ACHIEVEMENTS:
    - List all quantifiable achievements
    - Include numerical metrics (%, $, team size)
    - Note any awards or recognition
    
    RESPONSIBILITIES:
    - Management/leadership roles
    - Project ownership
    - Team collaboration
    - Strategic initiatives
    
    TECHNICAL ELEMENTS:
    - Tools and technologies used
    - Systems or processes improved
    - Technical projects completed
    
    FORMAT REQUIREMENTS:
    - Maintain chronological order
    - Preserve all numerical data
    - Keep original metrics and measurements
    - Include all location information
    
    OUTPUT STRUCTURE:
    {Company Name} ({Duration})
    {Title} - {Location}
    Achievements:
    [Bullet points with metrics]
    
    Key Responsibilities:
    [Bullet points]
    
    Technical Contributions:
    [Bullet points]
    
    ---
    
    Extract everything that matches this structure, maintaining the exact numbers, percentages, and metrics from the original resume.
    """
    return comprehensive_prompt

In [2]:
pip install pdfminer.six


Note: you may need to restart the kernel to use updated packages.


In [3]:
from pdfminer.high_level import extract_text
 
def extract_text_from_pdf(pdf_path):
    return extract_text(pdf_path)
 
if __name__ == '__main__':
    print(extract_text_from_pdf(r"ocr pdf.pdf"))

FIRST NAME LAST NAME
(XXX) XXX-XXXX | ProfessionalEmail@gmail.com | linkedin.com/in | City, State

EDUCATION
Degree, Major (e.g Bachelor of Science, Communication Studies)
Name of University, Institution
Minor: XXXXX    GPA x.x/4.0 (optional but recommend including if over a 3.0, can be Overall GPA or Major GPA)

Graduating Month Year
City, State

If you have a Masters Degree then repeat above format for undergraduate with your highest level of education appearing first
Do not use bullet points in the education section and do not include high school information

WORK EXPERIENCE (could also be titled Relevant Experience or Related Experience)
Position Title
Organization/Company Name
● Write your main highlighted accomplishments.
● Think about how your task/project helped the company do better and how you added value to the company.
● Follow the format “Performed X by doing Y resulting in Z”, quantify with numbers, percentages, other data where you can
● Start with strong action verbs an

In [8]:
from pdfminer.high_level import extract_text
 
def extract_text_from_pdf(pdf_path):
    return extract_text(pdf_path)
 
if __name__ == '__main__':
    print(extract_text_from_pdf(r"ocr pdf.pdf"))

FIRST NAME LAST NAME
(XXX) XXX-XXXX | ProfessionalEmail@gmail.com | linkedin.com/in | City, State

EDUCATION
Degree, Major (e.g Bachelor of Science, Communication Studies)
Name of University, Institution
Minor: XXXXX    GPA x.x/4.0 (optional but recommend including if over a 3.0, can be Overall GPA or Major GPA)

Graduating Month Year
City, State

If you have a Masters Degree then repeat above format for undergraduate with your highest level of education appearing first
Do not use bullet points in the education section and do not include high school information

WORK EXPERIENCE (could also be titled Relevant Experience or Related Experience)
Position Title
Organization/Company Name
● Write your main highlighted accomplishments.
● Think about how your task/project helped the company do better and how you added value to the company.
● Follow the format “Performed X by doing Y resulting in Z”, quantify with numbers, percentages, other data where you can
● Start with strong action verbs an

In [ ]:
d=extract_text_from_pdf(r"ocr pdf.pdf")

In [2]:
import pdfminer
import re

def extract_text_from_pdf(pdf_path):
    return extract_text(pdf_path)

def extract_name_from_resume(text):
    name = None

    # Use regex pattern to find a potential name
    pattern = r"(\b[A-Z][a-z]+\b)\s(\b[A-Z][a-z]+\b)"
    match = re.search(pattern, text)
    if match:
        name = match.group()

    return name

if __name__ == '__main__':
    text = extract_text_from_pdf(r"ocr pdf.pdf")
    name = extract_name_from_resume(text)

    if name:
        print("Name:", name)
    else:
        print("Name not found")

Name: Communication Studies


In [8]:
pip install --upgrade pdfminer.six


Note: you may need to restart the kernel to use updated packages.


In [3]:
def extract_text_from_pdf(pdf_path):
    return extract_text(pdf_path)

def extract_contact_number_from_resume(text):
    contact_number = None

    # Use regex pattern to find a potential contact number
    pattern = r"\b(?:\+?\d{1,3}[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}\b"
    match = re.search(pattern, text)
    if match:
        contact_number = match.group()

    return contact_number

if __name__ == '__main__':
    text = extract_text_from_pdf(r"ocr pdf.pdf")
    contact_number = extract_contact_number_from_resume(text)

    if contact_number:
        print("Contact Number:", contact_number)
    else:
        print("Contact Number not found")

Contact Number not found


In [4]:
def extract_text_from_pdf(pdf_path):
    return extract_text(pdf_path)

def extract_email_from_resume(text):
    email = None

    # Use regex pattern to find a potential email address
    pattern = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
    match = re.search(pattern, text)
    if match:
        email = match.group()

    return email

if __name__ == '__main__':
    text = extract_text_from_pdf(r"ocr pdf.pdf")
    email = extract_email_from_resume(text)

    if email:
        print("Email:", email)
    else:
        print("Email not found")

Email: ProfessionalEmail@gmail.com


In [5]:
def extract_text_from_pdf(pdf_path):
    return extract_text(pdf_path)

def extract_skills_from_resume(text, skills_list):
    skills = []

    # Search for skills in the resume text
    for skill in skills_list:
        pattern = r"\b{}\b".format(re.escape(skill))
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            skills.append(skill)

    return skills

if __name__ == '__main__':
    text = extract_text_from_pdf(r"ocr pdf.pdf")

    # List of predefined skills
    skills_list = ['Python', 'Data Analysis', 'Machine Learning', 'Communication', 'Project Management', 'Deep Learning', 'SQL', 'Tableau']

    extracted_skills = extract_skills_from_resume(text, skills_list)

    if extracted_skills:
        print("Skills:", extracted_skills)
    else:
        print("No skills found")

Skills: ['Communication', 'Project Management']


In [6]:
def extract_text_from_pdf(pdf_path):
    return extract_text(pdf_path)

def extract_education_from_resume(text):
    education = []

    # Use regex pattern to find education information
    pattern = r"(?i)(?:(?:Bachelor|B\.S\.|B\.A\.|Master|M\.S\.|M\.A\.|Ph\.D\.)\s(?:[A-Za-z]+\s)*[A-Za-z]+)"
    matches = re.findall(pattern, text)
    for match in matches:
        education.append(match.strip())

    return education

if __name__ == '__main__':
    text = extract_text_from_pdf(r"ocr pdf.pdf")

    extracted_education = extract_education_from_resume(text)
    if extracted_education:
        print("Education:", extracted_education)
    else:
        print("No education information found")

Education: ['Bachelor of Science']


In [7]:
def extract_text_from_pdf(pdf_path):
    return extract_text(pdf_path)

def extract_education_from_resume(text):
    education = []

    # Use regex pattern to find education information
    pattern = r"(?i)(?:Bsc|\bB\.\w+|\bM\.\w+|\bPh\.D\.\w+|\bBachelor(?:'s)?|\bMaster(?:'s)?|\bPh\.D)\s(?:\w+\s)*\w+"
    matches = re.findall(pattern, text)
    for match in matches:
        education.append(match.strip())

    return education

if __name__ == '__main__':
    text = extract_text_from_pdf(r"ocr pdf.pdf")

    extracted_education = extract_education_from_resume(text)
    if extracted_education:
        print("Education:", extracted_education)
    else:
        print("No education information found")

Education: ['Bachelor of Science']


In [9]:
pip install spacy

  Using cached typer-0.15.1-py3-none-any.whl.metadata (15 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 4.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 634.7/634.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.0/761.0 kB 5.6 MB/s eta 0:00:00
Using cached typer-0.15.1-py3-none-any.whl (44 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 8.7 MB/s eta 0:00:00a 0:00:01
Using cached shellingham-1.5.4-py2.py3-none-any.whl (9.8 kB)
Note: you may need to restart the kernel to use updated packages.


In [11]:
def extract_college_name(text):
    lines = text.split('\n')
    college_pattern = r"(?i).*college.*"
    for line in lines:
        if re.match(college_pattern, line):
            return line.strip()
    return None

# Example usage:
    text = extract_text_from_pdf(r"ocr pdf.pdf")


college_name = extract_college_name(text)
if college_name:
    print("College:", college_name)
else:
    print("College name not found.")

College name not found.


In [21]:
pip install resume-parser

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 1.4 MB/s eta 0:00:00a 0:00:010m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 4.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 4.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 2.9 MB/s eta 0:00:00a 0:00:01
  Created wheel for docx2txt: filename=docx2txt-0.8-py3-none-any.whl size=3960 sha256=9ee8bbf004dadeb3c73d1ecb22db24345c1676eb473730ca52472749a29be768
  Stored in directory: /Users/anju.sharma/Library/Caches/pip/wheels/6f/81/48/001bbc0109c15e18c009eee300022f42d1e070e54f1d00b218
  Created wheel for stemming: filename=stemming-1.0.1-py3-none-any.whl size=11123 sha256=b548de01c0fcf745a09468885a908bfdc1bb13013572c570ad0bbe521a9de9d6
  Stored in directory: /Users/anju.sharma/Library/Caches/pip/wheels/20/d4/73/028ca44cd75949ad81250dd3

In [ ]:
from pdfminer.high_level import extract_text
 
def extract_text_from_pdf(pdf_path):
    return extract_text(pdf_path)
 
if __name__ == '__main__':
    print(extract_text_from_pdf(r"JyantResume (1)-1-1.pdf"))

JYANT

M.Tech Student in Applied Mathematics and Scientific Computing 

E

 91 9053642807



jyant.2001@gmail.com

q

linkedin.com/in/Jyant-45417b25a/



Panipat, Haryana

J

SUMMARY

I am a dedicated student pursuing my M.Tech in Applied Mathematics and Scientific 
Computing, with a solid foundation in data analysis and machine learning. My academic 
journey includes qualifying prestigious exams like CSIR NET and GATE with high 
rankings. I possess strong analytical skills and hands-on experience in various technical 
projects focused on predictive modeling and data visualization.

EDUCATION

M.Tech.

IIT Roorkee

KEY ACHIEVEMENTS


CSIR NET Qualification

Qualified the CSIR NET JRF  Mathematical science 
2023 with AIR 184

GATE Mathematics Qualification

Qualified the GATE Mathematics 2023 with AIR 671

GATE Data Science Qualification

Qualified GATE DATA SCIENCE 2024 with AIR 3165

0

s

07/2023   05/2025 

Roorkee, Uttarakhand, India

SKILLS

M.Sc.

Kurukshetra University

07/20

In [2]:
pip install deepseek

Note: you may need to restart the kernel to use updated packages.


In [11]:
d

'FIRST NAME LAST NAME\n(XXX) XXX-XXXX | ProfessionalEmail@gmail.com | linkedin.com/in | City, State\n\nEDUCATION\nDegree, Major (e.g Bachelor of Science, Communication Studies)\nName of University, Institution\nMinor: XXXXX    GPA x.x/4.0 (optional but recommend including if over a 3.0, can be Overall GPA or Major GPA)\n\nGraduating Month Year\nCity, State\n\nIf you have a Masters Degree then repeat above format for undergraduate with your highest level of education appearing first\nDo not use bullet points in the education section and do not include high school information\n\nWORK EXPERIENCE (could also be titled Relevant Experience or Related Experience)\nPosition Title\nOrganization/Company Name\n● Write your main highlighted accomplishments.\n● Think about how your task/project helped the company do better and how you added value to the company.\n● Follow the format “Performed X by doing Y resulting in Z”, quantify with numbers, percentages, other data where you can\n● Start with s

In [ ]:
import pandas as pd
from deepseekri import RIModel

# Create a DataFrame from the resume variable
data = pd.DataFrame({'text': [d]})
model = RIModel('lama_mistral_model.h5')

# Define the query for extracting information
query = {
    'extract': ['Number of years of experience', 'Area of expertise', 'Specific methodology'],
}

# Extract the information using DeepSeek-RI
results = model.query(data, query)

# Print the results
for doc_id, result in results['documents']:
    print(f'Document ID: {doc_id}\n')
    for entity, value in result['results'].items():
        print(f"{entity}: {value}")

ModuleNotFoundError: No module named 'deepseekri'

In [ ]:
pip show deepseekri

Note: you may need to restart the kernel to use updated packages.


In [10]:
# import nltk
# from nltk.corpus import wordnet
# import re

# # Define a function to get the synsets for a given word
# def get_synsets(word):
#     synsets = []
#     for syn in wordnet.synsets(word, pos='n'):
#         synsets.append((syn.name(), syn))
#     return synsets

# # Define a function to extract information from the resume
# def extract_info(resume):
#     info = defaultdict(list)

#     # Split the text into sentences
#     sentences = nltk.sent_tokenize(resume)

#     for sentence in sentences:
#         # Tokenize the sentence into words
#         words = nltk.word_tokenize(sentence)

#         # Tag each word using part-of-speech tagging
#         pos_tags = nltk.pos_tag(words)

#         # Iterate through the words and their parts of speech
#         for word, pos in pos_tags:
#             if pos == 'NN' or pos == 'NNS':  # Extracting nouns
#                 synsets = get_synsets(word.lower())

#                 # Check if the synset contains a relevant concept (e.g., years, expertise)
#                 for synname, syn in synsets:
#                     if 'year' in synname or word.lower() == 'expertise':
#                         info['experience'].append(f"{word} ({syn.definition()[0]})")
#             elif re.match(r'^\d+(\.\d+)?$', word):  # Extracting numbers
#                 info['number_of_years'].append(word)
#             elif pos == 'VBG' and re.search(r'\bagile\b|scrum\b', word, re.IGNORECASE):  # Extracting methodologies (using regular expressions)
#                 info['methodology'].append(word)

#     return dict(info)

# # Load the resume text and extract information using NLTK with regular expressions
# # resume_text = "A job applicant's resume states they have 10 years of experience in software development and management, specializing in full-stack development and agile methodologies."

# a=extract_text_from_pdf(r"JyantResume (1)-1-1.pdf")
# result = extract_info(a)
# print(result)

{'number_of_years': ['9053642807', '2023', '184', '2023', '671', '2024', '3165', '0', '96']}


In [17]:
ats=extract_text_from_pdf(r"ResumeDTanyaDhingra.pdf")

In [14]:
c=extract_text_from_pdf(r"JyantResume (1)-1-1.pdf")

In [15]:
c

'JYANT\n\nM.Tech Student in Applied Mathematics and Scientific Computing \n\nE\n\n\x0091 9053642807\n\n\ue069\n\njyant.2001@gmail.com\n\nq\n\nlinkedin.com/in/Jyant-45417b25a/\n\n\ue092\n\nPanipat, Haryana\n\nJ\n\nSUMMARY\n\nI am a dedicated student pursuing my M.Tech in Applied Mathematics and Scientific \nComputing, with a solid foundation in data analysis and machine learning. My academic \njourney includes qualifying prestigious exams like CSIR NET and GATE with high \nrankings. I possess strong analytical skills and hands-on experience in various technical \nprojects focused on predictive modeling and data visualization.\n\nEDUCATION\n\nM.Tech.\n\nIIT Roorkee\n\nKEY ACHIEVEMENTS\n\ue015\n\nCSIR NET Qualification\n\nQualified the CSIR NET\x00JRF\x00 Mathematical science \n2023 with AIR 184\n\nGATE Mathematics Qualification\n\nQualified the GATE Mathematics 2023 with AIR 671\n\nGATE Data Science Qualification\n\nQualified GATE DATA SCIENCE 2024 with AIR 3165\n\n0\n\ns\n\n07/2023 \x00

In [11]:
ollama serve

SyntaxError: invalid syntax (2189434629.py, line 1)

In [12]:
from ollama import Client

# Initialize client - this will connect to the already running Ollama service
client = Client(host='http://localhost:11434')

# Simple test
response = client.chat(model='llama3.1:8b', messages=[
    {
        'role': 'user',
        'content': 'Hi, is this working?'
    }
])

print(f"Response: {response['message']['content']}")

Response: It seems like I've just started our conversation! Yes, it's working. How can I assist you today?


In [ ]:
from ollama import Client
import os

def process_prompts_from_file(filename, model_name='llama3.1:8b'):
    # Initialize Ollama client
    client = Client(host='http://localhost:11434')
    
    try:
        # Read the text file
        with open(filename, 'r', encoding='utf-8') as file:
            # Option 1: Process whole file as one prompt
            content = file.read()
            print(f"Processing prompt from file: {filename}")
            print("-" * 50)
            
            response = client.chat(model=model_name, messages=[
                {
                    'role': 'user',
                    'content': content
                }
            ])
            print(f"Response:\n{response['message']['content']}\n")
            
            # Option 2: Process file line by line
            print("Processing file line by line:")
            print("-" * 50)
            
            # Reset file pointer to beginning
            file.seek(0)
            for line_number, line in enumerate(file, 1):
                # Skip empty lines
                line = line.strip()
                if not line:
                    continue
                    
                print(f"\nPrompt {line_number}: {line}")
                response = client.chat(model=model_name, messages=[
                    {
                        'role': 'user',
                        'content': line
                    }
                ])
                print(f"Response: {response['message']['content']}")
                
    except FileNotFoundError:
        print(f"Error: File '{filename}' not found")
    except Exception as e:
        print(f"An error occurred: {str(e)}")

# Example usage
if __name__ == "__main__":
    # Create a sample prompts file
    sample_prompts = """What is artificial intelligence?
    Explain quantum computing in simple terms.
    Write a haiku about programming."""
    
    # Write sample prompts to a file
    with open('prompts.txt', 'w', encoding='utf-8') as f:
        f.write(sample_prompts)
    
    # Process the prompts
    process_prompts_from_file('prompts.txt')

In [13]:
from ollama import Client

def process_text(text, model_name='llama3.1:8b', process_by_line=False):
    # Initialize Ollama client
    client = Client(host='http://localhost:11434')
    
    try:
        # Option 1: Process entire text at once
        if not process_by_line:
            print("Processing entire text:")
            print("-" * 50)
            response = client.chat(model=model_name, messages=[
                {
                    'role': 'user',
                    'content': text
                }
            ])
            print(f"Response:\n{response['message']['content']}\n")
            return response['message']['content']
            
        # Option 2: Process text line by line
        else:
            print("Processing text line by line:")
            print("-" * 50)
            responses = []
            
            # Split text into lines and process each non-empty line
            for line_number, line in enumerate(text.split('\n'), 1):
                line = line.strip()
                if not line:
                    continue
                    
                print(f"\nPrompt {line_number}: {line}")
                response = client.chat(model=model_name, messages=[
                    {
                        'role': 'user',
                        'content': line
                    }
                ])
                print(f"Response: {response['message']['content']}")
                responses.append(response['message']['content'])
            
            return responses
                
    except Exception as e:
        print(f"An error occurred: {str(e)}")
        return None

# Example usage
if __name__ == "__main__":
    # Example text stored in a variable
    text_content = """What is artificial intelligence?
    Explain quantum computing in simple terms.
    Write a haiku about programming."""
    
    # Process entire text at once
    print("Processing as single prompt:")
    result = process_text(text_content, process_by_line=False)
    
    print("\n" + "="*50 + "\n")
    
    # Process text line by line
    print("Processing line by line:")
    results = process_text(text_content, process_by_line=True)

Processing as single prompt:
Processing entire text:
--------------------------------------------------
Response:
Here are the answers to your questions:

**What is Artificial Intelligence?**

Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks that would typically require human intelligence, such as:

* Learning from data and experiences
* Solving complex problems
* Recognizing patterns and making decisions
* Interacting with humans in a natural way

AI systems use algorithms and data to make predictions, classify objects, or generate text. There are several types of AI, including:

* Narrow or Weak AI: designed to perform a specific task, such as playing chess or recognizing faces.
* General or Strong AI: capable of performing any intellectual task that humans can.

**Quantum Computing in Simple Terms**

Imagine you have a safe with many combinations. A classical computer would try each combination one by one, but a quantum computer is li

In [14]:
from ollama import Client

def analyze_text(text, model_name='llama3.1:8b'):
    # Initialize Ollama client
    client = Client(host='http://localhost:11434')
    
    # Remove any extra whitespace and ensure text is clean
    text = text.strip()
    
    # Different types of analysis prompts
    prompts = [
        f"Please summarize this text in 2-3 sentences: {text}",
        f"What are the main topics or themes in this text: {text}",
        f"What are the key points made in this text: {text}",
        f"Generate 3 relevant questions about this text: {text}",
        f"Analyze the tone and style of this text: {text}"
    ]
    
    print("Analyzing text:", text[:100] + "..." if len(text) > 100 else text)
    print("\nRunning analysis...")
    
    # Process each prompt
    results = {}
    for prompt in prompts:
        try:
            response = client.chat(model=model_name, messages=[
                {
                    'role': 'user',
                    'content': prompt
                }
            ])
            # Store the result with a descriptive key
            key = prompt.split(':')[0].strip()
            results[key] = response['message']['content']
            
        except Exception as e:
            print(f"Error processing prompt: {str(e)}")
    
    return results

def custom_analysis(text, custom_prompt, model_name='llama2'):
    """Function to analyze text with a custom prompt"""
    client = Client(host='http://localhost:11434')
    
    try:
        full_prompt = f"{custom_prompt}: {text}"
        response = client.chat(model=model_name, messages=[
            {
                'role': 'user',
                'content': full_prompt
            }
        ])
        return response['message']['content']
        
    except Exception as e:
        print(f"Error processing prompt: {str(e)}")
        return None

# Example usage
if __name__ == "__main__":
    # Sample text
    sample_text = """
    Machine learning is a subset of artificial intelligence that focuses on developing 
    systems that can learn from and make decisions based on data. It has numerous 
    applications in various fields, from healthcare to finance.
    """
    
    # Run standard analysis
    print("Running standard analysis...")
    results = analyze_text(sample_text)
    
    # Print results
    print("\nAnalysis Results:")
    print("=" * 50)
    for key, value in results.items():
        print(f"\n{key}:")
        print(value)
    
    # Example of custom prompt
    print("\nCustom Analysis:")
    print("=" * 50)
    custom_question = "What are potential future applications of the technology mentioned in this text?"
    custom_result = custom_analysis(sample_text, custom_question)
    print(f"\nResponse to custom prompt:")
    print(custom_result)

Running standard analysis...
Analyzing text: Machine learning is a subset of artificial intelligence that focuses on developing 
    systems that...

Running analysis...

Analysis Results:

Please summarize this text in 2-3 sentences:
Here is a summary of the text in 2-3 sentences:

Machine learning is a type of artificial intelligence that allows systems to learn from and make decisions based on data. This technology has many practical uses across different industries, including healthcare and finance. Machine learning enables systems to improve their performance over time through experience and data analysis.

What are the main topics or themes in this text:
The two main topics or themes in this text are:

1. **Machine Learning**: The concept itself and its definition as a subset of Artificial Intelligence.
2. **Applications of Machine Learning**: The various fields where machine learning is utilized, such as healthcare and finance.

Let me know if you'd like me to help with anything

In [16]:
from ollama import Client

def analyze_text(text, model_name='llama3.1:8b'):
    # Initialize Ollama client
    client = Client(host='http://localhost:11434')
    
    # Remove any extra whitespace and ensure text is clean
    text = text.strip()
    
    # Different types of analysis prompts
    prompts = [
        f"Please find experince in one line: {text}",
        f"What is the name of the resumer: {text}",
        f"What are the education level of the resumer: {text}",
        f"extract knowledge skills of the resumer: {text}"
       
    ]
    
    print("Analyzing text:", text[:100] + "..." if len(text) > 100 else text)
    print("\nRunning analysis...")
    
    # Process each prompt
    results = {}
    for prompt in prompts:
        try:
            response = client.chat(model=model_name, messages=[
                {
                    'role': 'user',
                    'content': prompt
                }
            ])
            # Store the result with a descriptive key
            key = prompt.split(':')[0].strip()
            results[key] = response['message']['content']
            
        except Exception as e:
            print(f"Error processing prompt: {str(e)}")
    
    return results

def custom_analysis(text, custom_prompt, model_name='llama3.1:8b'):
    """Function to analyze text with a custom prompt"""
    client = Client(host='http://localhost:11434')
    
    try:
        full_prompt = f"{custom_prompt}: {text}"
        response = client.chat(model=model_name, messages=[
            {
                'role': 'user',
                'content': full_prompt
            }
        ])
        return response['message']['content']
        
    except Exception as e:
        print(f"Error processing prompt: {str(e)}")
        return None

# Example usage
if __name__ == "__main__":
    # Sample text
    sample_text = c
    
    # Run standard analysis
    print("Running standard analysis...")
    results = analyze_text(sample_text)
    
    # Print results
    print("\nAnalysis Results:")
    print("=" * 50)
    for key, value in results.items():
        print(f"\n{key}:")
        print(value)
    


Running standard analysis...
Analyzing text: JYANT

M.Tech Student in Applied Mathematics and Scientific Computing 

E

 91 9053642807



jyant....

Running analysis...

Analysis Results:

Please find experince in one line:
Here is JYANT's experience in one line:

M.Tech Student in Applied Mathematics and Scientific Computing with hands-on experience in predictive modeling, data visualization, and machine learning projects.

What is the name of the resumer:
The resume is named: **JYANT**

What are the education level of the resumer:
The education level of JYANT can be inferred from his resume as follows:

1. **10th Grade**: JYANT completed his 10th grade from the Haryana Board of School Education in 2015.
2. **12th Grade**: He then pursued and completed his 12th grade from the same board in 2017.
3. **B.Sc. Non-Medical**: JYANT earned a Bachelor of Science (Non-Medical) degree from Kurukshetra University in 2020.
4. **M.Sc.**: He then went on to pursue and complete his Master of Scien

In [19]:
from ollama import Client

def analyze_text(text, model_name='llama3.1:8b'):
    # Initialize Ollama client
    client = Client(host='http://localhost:11434')
    
    # Remove any extra whitespace and ensure text is clean
    text = text.strip()
    
    # Different types of analysis prompts
    prompts = [
        f"Please find experince in one line: {text}",
        f"What is the name of the resumer: {text}",
        f"What are the education level of the resumer: {text}",
        f"extract knowledge skills of the resumer: {text}",
        f"get me the contact details of the resumer:{text}"
       
    ]
    
    print("Analyzing text:", text[:100] + "..." if len(text) > 100 else text)
    print("\nRunning analysis...")
    
    # Process each prompt
    results = {}
    for prompt in prompts:
        try:
            response = client.chat(model=model_name, messages=[
                {
                    'role': 'user',
                    'content': prompt
                }
            ])
            # Store the result with a descriptive key
            key = prompt.split(':')[0].strip()
            results[key] = response['message']['content']
            
        except Exception as e:
            print(f"Error processing prompt: {str(e)}")
    
    return results

def custom_analysis(text, custom_prompt, model_name='llama3.1:8b'):
    """Function to analyze text with a custom prompt"""
    client = Client(host='http://localhost:11434')
    
    try:
        full_prompt = f"{custom_prompt}: {text}"
        response = client.chat(model=model_name, messages=[
            {
                'role': 'user',
                'content': full_prompt
            }
        ])
        return response['message']['content']
        
    except Exception as e:
        print(f"Error processing prompt: {str(e)}")
        return None

# Example usage
if __name__ == "__main__":
    # Sample text
    sample_text = ats
    
    # Run standard analysis
    print("Running standard analysis...")
    results = analyze_text(sample_text)
    
    # Print results
    print("\nAnalysis Results:")
    print("=" * 50)
    for key, value in results.items():
        print(f"\n{key}:")
        print(value)
    


Running standard analysis...
Analyzing text: TANYA DHINGRA 

Delhi, India | +91-8901128354 | Tanyadhingra186@gmail.com | WWW: https://www.linkedi...

Running analysis...

Analysis Results:

Please find experince in one line:
Here is a one-line experience summary:

Highly detail-oriented and analytical data professional with 2+ years of experience in data analysis, visualization, and automation using tools like Tableau, Power BI, Python, R, and SQL, driving business outcomes through actionable insights.

What is the name of the resumer:
The resume belongs to Tanya Dhingra.

What are the education level of the resumer:
Here's an analysis of Tanya Dhingra's education level:

**Master of Computer Applications (M.C.A.) - Software Engineering**

* University: University School of Information, Communication and Technology, GGSIPU
* Duration: 10/2021 to 07/2023
* GPA: 8.97

This is a master's degree in computer applications with a specialization in software engineering.

**Bachelor of Science 

In [4]:
ats=extract_text_from_pdf(r"ResumeDTanyaDhingra.pdf")

In [9]:
from ollama import Client

def analyze_text(text, model_name='llama3.2:latest'):
    # Initialize Ollama client
    client = Client(host='http://localhost:11434')
    
    # Remove any extra whitespace and ensure text is clean
    text = text.strip()
    
    # Different types of analysis prompts
    prompts = [
        f"Please find experince in one line: {text}",
        f"What is the name of the resumer: {text}",
        f"What are the education level of the resumer: {text}",
        f"extract knowledge skills of the resumer: {text}",
        f"get me the contact details of the resumer:{text}"
       
    ]
    
    print("Analyzing text:", text[:100] + "..." if len(text) > 100 else text)
    print("\nRunning analysis...")
    
    # Process each prompt
    results = {}
    for prompt in prompts:
        try:
            response = client.chat(model=model_name, messages=[
                {
                    'role': 'user',
                    'content': prompt
                }
            ])
            # Store the result with a descriptive key
            key = prompt.split(':')[0].strip()
            results[key] = response['message']['content']
            
        except Exception as e:
            print(f"Error processing prompt: {str(e)}")
    
    return results

def custom_analysis(text, custom_prompt, model_name='llama3.2:latest'):
    """Function to analyze text with a custom prompt"""
    client = Client(host='http://localhost:11434')
    
    try:
        full_prompt = f"{custom_prompt}: {text}"
        response = client.chat(model=model_name, messages=[
            {
                'role': 'user',
                'content': full_prompt
            }
        ])
        return response['message']['content']
        
    except Exception as e:
        print(f"Error processing prompt: {str(e)}")
        return None

# Example usage
if __name__ == "__main__":
    # Sample text
    sample_text = ats
    
    # Run standard analysis
    print("Running standard analysis...")
    results = analyze_text(sample_text)
    
    # Print results
    print("\nAnalysis Results:")
    print("=" * 50)
    for key, value in results.items():
        print(f"\n{key}:")
        print(value)
    


Running standard analysis...
Analyzing text: TANYA DHINGRA 

Delhi, India | +91-8901128354 | Tanyadhingra186@gmail.com | WWW: https://www.linkedi...

Running analysis...

Analysis Results:

Please find experince in one line:
Tanya Dhindra is a Data Analyst with experience in data analysis, Python development, and project management, having worked for PhysicsWallah, iTechplement, MedTourEasy, and developed projects such as anomaly detection for Metaverse dataset, Amazon global dashboard using Power BI, Resource Tracking Tool, Wine Quality Analysis using MCDM, Skittles Website/App Development, COVID-19 Impact Analysis using Python, Retrieval-Augmented Generation (RAG) Based Search Tool.

What is the name of the resumer:
The resume belongs to TANYA DHINGRA.

What are the education level of the resumer:
Based on the provided resume, here is an education level breakdown for Tanya Dhinagra:

1. Bachelor of Science (B.Sc.) in Computer Science: 7.95 GPA, completed from July 2018 to July 2021 a

In [11]:
from ollama import Client

def analyze_text(text, model_name='llama3.1:8b'):
    # Initialize Ollama client
    client = Client(host='http://localhost:11434')
    
    # Remove any extra whitespace and ensure text is clean
    text = text.strip()
    
    # Different types of analysis prompts
    prompts = [
        f"Please find experince in one line: {text}",
        f"What is the name of the resumer: {text}",
        f"What are the education level of the resumer: {text}",
        f"extract knowledge skills of the resumer: {text}",
        f"get me the contact details of the resumer:{text}"
       
    ]
    
    print("Analyzing text:", text[:100] + "..." if len(text) > 100 else text)
    print("\nRunning analysis...")
    
    # Process each prompt
    results = {}
    for prompt in prompts:
        try:
            response = client.chat(model=model_name, messages=[
                {
                    'role': 'user',
                    'content': prompt
                }
            ])
            # Store the result with a descriptive key
            key = prompt.split(':')[0].strip()
            results[key] = response['message']['content']
            
        except Exception as e:
            print(f"Error processing prompt: {str(e)}")
    
    return results

def custom_analysis(text, custom_prompt, model_name='llama3.1:8b'):
    """Function to analyze text with a custom prompt"""
    client = Client(host='http://localhost:11434')
    
    try:
        full_prompt = f"{custom_prompt}: {text}"
        response = client.chat(model=model_name, messages=[
            {
                'role': 'user',
                'content': full_prompt
            }
        ])
        return response['message']['content']
        
    except Exception as e:
        print(f"Error processing prompt: {str(e)}")
        return None

# Example usage
if __name__ == "__main__":
    # Sample text
    sample_text = ats
    
    # Run standard analysis
    print("Running standard analysis...")
    results = analyze_text(sample_text)
    
    # Print results
    print("\nAnalysis Results:")
    print("=" * 50)
    for key, value in results.items():
        print(f"\n{key}:")
        print(value)
    


Running standard analysis...
Analyzing text: TANYA DHINGRA 

Delhi, India | +91-8901128354 | Tanyadhingra186@gmail.com | WWW: https://www.linkedi...

Running analysis...

Analysis Results:

Please find experince in one line:
Here is Tanya Dhingra's experience in one line:

Data Analyst and Python Developer with 3+ years of experience in data analysis, visualization, and machine learning, having worked on various projects including anomaly detection, resource tracking tool development, and COVID-19 impact analysis.

What is the name of the resumer:
The resumer is: **TANYA DHINGRA**.

What are the education level of the resumer:
Here's a summary of the education level and qualifications of Tanya Dhingra:

**Education Level:** Postgraduate

**Qualifications:**

1. **Post Graduate Program in Data Science Engineering - AI & ML**, Great Lakes Institute of Management, 2024-2025
2. **Master of Computer Applications (M.C.A.)-Software Engineering**, University School of Information, Communicatio

In [12]:
from ollama import Client

def analyze_text(text, model_name='qwen2.5-coder:1.5b'):
    # Initialize Ollama client
    client = Client(host='http://localhost:11434')
    
    # Remove any extra whitespace and ensure text is clean
    text = text.strip()
    
    # Different types of analysis prompts
    prompts = [
        f"Please find experince in one line: {text}",
        f"What is the name of the resumer: {text}",
        f"What are the education level of the resumer: {text}",
        f"extract knowledge skills of the resumer: {text}",
        f"get me the contact details of the resumer:{text}"
       
    ]
    
    print("Analyzing text:", text[:100] + "..." if len(text) > 100 else text)
    print("\nRunning analysis...")
    
    # Process each prompt
    results = {}
    for prompt in prompts:
        try:
            response = client.chat(model=model_name, messages=[
                {
                    'role': 'user',
                    'content': prompt
                }
            ])
            # Store the result with a descriptive key
            key = prompt.split(':')[0].strip()
            results[key] = response['message']['content']
            
        except Exception as e:
            print(f"Error processing prompt: {str(e)}")
    
    return results

def custom_analysis(text, custom_prompt, model_name='qwen2.5-coder:1.5b'):
    """Function to analyze text with a custom prompt"""
    client = Client(host='http://localhost:11434')
    
    try:
        full_prompt = f"{custom_prompt}: {text}"
        response = client.chat(model=model_name, messages=[
            {
                'role': 'user',
                'content': full_prompt
            }
        ])
        return response['message']['content']
        
    except Exception as e:
        print(f"Error processing prompt: {str(e)}")
        return None

# Example usage
if __name__ == "__main__":
    # Sample text
    sample_text = ats
    
    # Run standard analysis
    print("Running standard analysis...")
    results = analyze_text(sample_text)
    
    # Print results
    print("\nAnalysis Results:")
    print("=" * 50)
    for key, value in results.items():
        print(f"\n{key}:")
        print(value)
    


Running standard analysis...
Analyzing text: TANYA DHINGRA 

Delhi, India | +91-8901128354 | Tanyadhingra186@gmail.com | WWW: https://www.linkedi...

Running analysis...

Analysis Results:

Please find experince in one line:
TANYA DHINGRA is a highly skilled and experienced data analyst with over 3 years of experience in the field. She has a Master's degree in Data Science Engineering, AI & ML from Great Lakes Institute of Management and completed various certifications such as Google Data Analytics Professional Certification and IBM Data Science Professional Certificate (Coursera). She specializes in Python for data analysis and machine learning and has experience with Tableau and Power BI for visualizations. Her analytical skills, problem-solving abilities, and project management expertise make her a valuable asset to any organization.

What is the name of the resumer:
The name of the resumer is TANYA DHINGRA.

What are the education level of the resumer:
Based on the resume provided

In [16]:
from ollama import Client

def analyze_text(text, model_name='llama3.1:8b'):
    # Initialize Ollama client
    client = Client(host='http://localhost:11434')
    
    # Remove any extra whitespace and ensure text is clean
    text = text.strip()
    
    # Different types of analysis prompts
    prompts = [
        f"Please find experince in one line: {text}",
        f"What is the name of the resumer: {text}",
        f"What are the education level of the resumer: {text}",
        f"extract knowledge skills of the resumer: {text}",
        f"get me the contact details of the resumer:{text}"
       
    ]
    
    print("Analyzing text:", text[:100] + "..." if len(text) > 100 else text)
    print("\nRunning analysis...")
    
    # Process each prompt
    results = {}
    for prompt in prompts:
        try:
            response = client.chat(model=model_name, messages=[
                {
                    'role': 'user',
                    'content': prompt
                }
            ])
            # Store the result with a descriptive key
            key = prompt.split(':')[0].strip()
            results[key] = response['message']['content']
            
        except Exception as e:
            print(f"Error processing prompt: {str(e)}")
    
    return results

def custom_analysis(text, custom_prompt, model_name='llama3.1:8b'):
    """Function to analyze text with a custom prompt"""
    client = Client(host='http://localhost:11434')
    
    try:
        full_prompt = f"{custom_prompt}: {text}"
        response = client.chat(model=model_name, messages=[
            {
                'role': 'user',
                'content': full_prompt
            }
        ])
        return response['message']['content']
        
    except Exception as e:
        print(f"Error processing prompt: {str(e)}")
        return None

# Example usage
if __name__ == "__main__":
    # Sample text
    sample_text = c
    
    # Run standard analysis
    print("Running standard analysis...")
    results = analyze_text(sample_text)
    
    # Print results
    print("\nAnalysis Results:")
    print("=" * 50)
    for key, value in results.items():
        print(f"\n{key}:")
        print(value)
    


Running standard analysis...
Analyzing text: JYANT

M.Tech Student in Applied Mathematics and Scientific Computing 

E

 91 9053642807



jyant....

Running analysis...

Analysis Results:

Please find experince in one line:
Here is the experience of Jyant in one line:

M.Tech Student in Applied Mathematics and Scientific Computing at IIT Roorkee, with hands-on experience in predictive modeling, data visualization, and machine learning through various technical projects.

What is the name of the resumer:
The name of the resume is: JYANT 

It appears that this is a resume for a student named Jyant, who is pursuing an M.Tech in Applied Mathematics and Scientific Computing from IIT Roorkee.

What are the education level of the resumer:
Based on the resume, here is an analysis of JYANT's education level:

**Highest Education Level:** M.Tech (Master of Technology) from IIT Roorkee in Applied Mathematics and Scientific Computing.

**Previous Education Levels:**

1. **M.Sc. (Master of Science

In [28]:
from ollama import Client
import json
import re

def analyze_resume(text, model_name='llama3.1:8b'):
    """Analyze resume text and return structured JSON data"""
    client = Client(host='http://localhost:11434')
    
    # Structured prompt for JSON output
    prompt = f"""Extract the following information from this resume in JSON format:
    {text}
    
    Return JSON with these keys:
    - "first_name" (string)
    -"last_name"(string)
    - "email" (string)
    - "phone_no" (number)
    - "education" (array of strings)
    - "passing out date (date) with course mapped correctly so that we can use this to fill in form"
    - "skills" (array of strings)
    - "experience" (array of strings with job titles and durations(format dd-mm-yy))
    - "certifications" (array of strings, optional)
    
    Format: {{ "key": "value" }} without any additional text."""

    try:
        response = client.chat(model=model_name, messages=[
            {'role': 'user', 'content': prompt}
        ])
        
        # Extract JSON from response
        raw_output = response['message']['content']
        
        # Use regex to find JSON in the response
        json_match = re.search(r'\{.*\}', raw_output, re.DOTALL)
        if json_match:
            json_str = json_match.group()
            return json.loads(json_str)
        
        return {"error": "No JSON found in response"}
    
    except json.JSONDecodeError:
        print("Error decoding JSON response")
        return {"error": "Invalid JSON format"}
    except Exception as e:
        print(f"Error processing request: {str(e)}")
        return {"error": str(e)}

# Example usage
if __name__ == "__main__":
    # sample_resume = """
    # John Doe
    # Email: john.doe@email.com | Phone: (555) 123-4567
    # Education: 
    # - BSc Computer Science, University of Example (2020-2024)
    # Skills: Python, Machine Learning, SQL, Data Analysis
    # Experience:
    # - Data Scientist at XYZ Corp (2022-present)
    # - ML Intern at ABC Tech (2021)
    # Certifications: AWS Certified, TensorFlow Developer
    # """
    
    print("Analyzing resume...")
    result = analyze_resume(ats)
    
    print("\nStructured Resume Data:")
    print(json.dumps(result, indent=2))

Analyzing resume...

Structured Resume Data:
{
  "first_name": "Tanya",
  "last_name": "Dhingra",
  "email": "tanyadhingra186@gmail.com",
  "phone_no": "+919901128354",
  "education": [
    {
      "degree": "Master of Computer Applications (M.C.A.)-Software Engineering",
      "institution": "University School of Information, Communication and Technology, GGSIPU",
      "passing_out_date": "10/2021",
      "gpa": "8.97"
    },
    {
      "degree": "Post Graduate Program in Data Science Engineering - AI & ML",
      "institution": "Great Lakes Institute of Management",
      "passing_out_date": "01/2025",
      "duration": "09/2022 to 01/2025"
    },
    {
      "degree": "Bachelor of Science (B.Sc.) - Computer Science",
      "institution": "Maitreyi College, Delhi University",
      "passing_out_date": "07/2021",
      "gpa": "7.95"
    }
  ],
  "skills": [
    "Tableau",
    "Power BI",
    "MySQL",
    "MongoDB",
    "IBM OpenPages",
    "MS Office Suite (Excel, PowerPoint, Access

In [31]:
from ollama import Client
import json
import re

def analyze_resume(text, model_name='llama3.1:8b'):
    """Analyze resume text and return structured JSON data"""
    client = Client(host='http://localhost:11434')
    
    # Structured prompt for JSON output
    prompt = f"""Extract the following information from this resume in JSON format:
    {text}
    
    Return JSON with these keys:
    - "first_name" (string)
    -"last_name"(string)
    - "email" (string)
    - "phone_no" (number)
    -"adress (string) so that we can use this to fill in form"
    - "education" (array of strings)
    - "passing out date (date) with course mapped correctly so that we can use this to fill in form"
    - "skills" (array of strings)
    - "experience" (array of strings with job titles and durations(format dd-mm-yy) mapped correctly so that we can use this to fill in form)
    - "certifications" (array of strings, optional)
    -"main projects to map in the form (strings) with little bit description"
    
    Format: {{ "key": "value" }} without any additional text."""

    try:
        response = client.chat(model=model_name, messages=[
            {'role': 'user', 'content': prompt}
        ])
        
        # Extract JSON from response
        raw_output = response['message']['content']
        
        # Use regex to find JSON in the response
        json_match = re.search(r'\{.*\}', raw_output, re.DOTALL)
        if json_match:
            json_str = json_match.group()
            return json.loads(json_str)
        
        return {"error": "No JSON found in response"}
    
    except json.JSONDecodeError:
        print("Error decoding JSON response")
        return {"error": "Invalid JSON format"}
    except Exception as e:
        print(f"Error processing request: {str(e)}")
        return {"error": str(e)}

# Example usage
if __name__ == "__main__":

    
    print("Analyzing resume...")
    result = analyze_resume(ats)
    
    print("\nStructured Resume Data:")
    print(json.dumps(result, indent=2))

Analyzing resume...

Structured Resume Data:
{
  "first_name": "Tanya",
  "last_name": "Dhingra",
  "email": "tanyadhingra186@gmail.com",
  "phone_no": "+91-8901128354",
  "address": "Delhi, India",
  "education": [
    {
      "course": "Post Graduate Program in Data Science Engineering - AI & ML",
      "university": "Great Lakes Institute of Management",
      "passing_out_date": "01/2025"
    },
    {
      "course": "Master of Computer Applications (M.C.A.)-Software Engineering",
      "university": "University School of Information, Communication and Technology, GGSIPU",
      "gpa": "8.97",
      "passing_out_date": "07/2023"
    },
    {
      "course": "Bachelor of Science (B.Sc.) - Computer Science",
      "university": "Maitreyi College, Delhi University",
      "gpa": "7.95",
      "passing_out_date": "07/2021"
    }
  ],
  "skills": [
    "Tableau",
    "Power BI",
    "MySQL",
    "MongoDB",
    "IBM OpenPages",
    "MS Office Suite (Excel, PowerPoint, Access)",
    "Pyth

In [41]:
ats

"TANYA DHINGRA \n\nDelhi, India | +91-8901128354 | Tanyadhingra186@gmail.com | WWW: https://www.linkedin.com/in/tanyadhingra13/ \n\nWorking Experience \n\n  Data Analyst, PhysicsWallah \n\n                                           09/2022 to 01/2025 \n          Delhi, India \n\n•  Analyzed datasets with 10,000+ records, improving decision-making by 15% through actionable insights. \n•  Developed automated dashboards in Tableau and Power BI, reducing manual reporting time by 30%. \n\nPython Developer Intern, iTechplement    \n\n          01/2024 to 02/2024 \n          Delhi, India \n\n•  Developed a command-line tool to extract, manipulate, and parse PDFs, enhancing document management efficiency.  \n•  Automated PDF extraction and data processing, reducing manual efforts by 40%. \n•  Utilized libraries like PyPDF2 and PDFMiner to build robust functionality for text extraction, metadata retrieval, and file conversion. \n\nData Analyst Intern, MedTourEasy \n\n          02/2021 to 04/202

In [32]:
ats_json={
  "first_name": "Tanya",
  "last_name": "Dhingra",
  "email": "tanyadhingra186@gmail.com",
  "phone_no": "+91-8901128354",
  "address": "Delhi, India",
  "education": [
    {
      "course": "Post Graduate Program in Data Science Engineering - AI & ML",
      "university": "Great Lakes Institute of Management",
      "passing_out_date": "01/2025"
    },
    {
      "course": "Master of Computer Applications (M.C.A.)-Software Engineering",
      "university": "University School of Information, Communication and Technology, GGSIPU",
      "gpa": "8.97",
      "passing_out_date": "07/2023"
    },
    {
      "course": "Bachelor of Science (B.Sc.) - Computer Science",
      "university": "Maitreyi College, Delhi University",
      "gpa": "7.95",
      "passing_out_date": "07/2021"
    }
  ],
  "skills": [
    "Tableau",
    "Power BI",
    "MySQL",
    "MongoDB",
    "IBM OpenPages",
    "MS Office Suite (Excel, PowerPoint, Access)",
    "Python",
    "R",
    "SQL"
  ],
  "experience": [
    {
      "job_title": "Data Analyst",
      "company": "PhysicsWallah",
      "duration": "09/2022 - 01/2025"
    },
    {
      "job_title": "Python Developer Intern",
      "company": "iTechplement",
      "duration": "01/2024 - 02/2024"
    },
    {
      "job_title": "Data Analyst Intern",
      "company": "MedTourEasy",
      "duration": "02/2021 - 04/2021"
    }
  ],
  "certifications": [
    "Google Data Analytics Professional Certification",
    "Alteryx Designer Core Certification (Ongoing)",
    "IBM Data Science Professional Certificate (Coursera)"
  ],
  "projects": [
    {
      "project_name": "Anomaly detection for Metaverse dataset",
      "description": "Analyzed metaverse transactional data using Python, machine learning models, and statistical techniques to predict user engagement, achieving high accuracy and actionable insights for improved retention."
    },
    {
      "project_name": "Amazon global dashboard using POWER BI",
      "description": "Built a Power BI dashboard for Amazon's global operations, analyzing sales, profits, churn, and returns, enabling real-time insights and actionable strategies to boost performance."
    },
    {
      "project_name": "Resource Tracking Tool",
      "description": "Streamlined resource management and time-tracking using Alteryx."
    },
    {
      "project_name": "Wine Quality Analysis using MCDM",
      "description": "Implemented Multi-Criteria Decision Making (MCDM) algorithms to evaluate wine quality, analyzing 1,599 samples based on 11 chemical features."
    },
    {
      "project_name": "Skittles Website/App Development",
      "description": "Led the full-stack development of a web/app platform for Skittles, improving user engagement by 30% through enhanced UI/UX design and 10+ interactive features."
    },
    {
      "project_name": "COVID-19 Impact Analysis using Python",
      "description": "Analyzed 2 lakh+ data points to study COVID-19's impact across regions, employing Python libraries (Pandas, Matplotlib) to visualize trends and derive actionable insights."
    },
    {
      "project_name": "Retrieval-Augmented Generation (RAG) Based Search Tool",
      "description": "Developed a RAG-based search tool, enhancing retrieval accuracy by 25% and improving response relevance across 10,000+ query samples."
    }
  ]
}

In [40]:
import streamlit as st

# Your data
passing_out_dates = {
    'M.C.A.': '07-2023',
    'PGP in Data Science Engineering': '01-2025',
    'B.Sc. Computer Science': '07-2021'
}

# Streamlit UI
st.title("Passing Out Dates Dashboard 🎓")

# Option 1: Display all data
st.header("All Programs")
st.write(passing_out_dates)

# Option 2: Let users select a program to view its date
st.header("Search by Program")
selected_program = st.selectbox(
    "Choose a program:",
    options=list(passing_out_dates.keys())
)

# Display the selected program's date
selected_date = passing_out_dates[selected_program]
st.success(f"**{selected_program}** passing out date: **{selected_date}**")

2025-02-15 01:40:02.354 
  command:

    streamlit run /opt/anaconda3/lib/python3.12/site-packages/ipykernel_launcher.py [ARGUMENTS]
2025-02-15 01:40:02.355 Session state does not function when running a script without `streamlit run`


DeltaGenerator()

In [24]:
js={
  "first_name": "Tanya",
  "last_name": "Dhingra",
  "email": "Tanyadhingra186@gmail.com",
  "phone_no": "+91-8901128354",
  "education": [
    "Post Graduate Program in Data Science Engineering - AI & ML, Great Lakes Institute of Management (02/2024 to 01/2025)",
    "Master of Computer Applications (M.C.A.)-Software Engineering GPA: 8.97, University School of Information, Communication and Technology, GGSIPU (10/2021 to 07/2023)",
    "Bachelor of Science (B.Sc.) - Computer Science GPA: 7.95, Maitreyi College, Delhi University (07/2018 to 07/2021)"
  ],
  "passing_out_date": {
    "M.C.A.": "07-2023",
    "PGP in Data Science Engineering": "01-2025",
    "B.Sc. Computer Science": "07-2021"
  },
  "skills": [
    "Tableau",
    "Power BI",
    "MySQL",
    "MongoDB",
    "IBM OpenPages",
    "MS Office Suite (Excel, PowerPoint, Access)",
    "Python",
    "R",
    "SQL",
    "Statistical Analysis",
    "Data Visualization",
    "Data Analysis"
  ],
  "experience": [
    "Data Analyst, PhysicsWallah (09/2022 to 01/2025)",
    "Python Developer Intern, iTechplement (01/2024 to 02/2024)",
    "Data Analyst Intern, MedTourEasy (02/2021 to 04/2021)"
  ],
  "certifications": [
    "Google Data Analytics Professional Certification",
    "Alteryx Designer Core Certification (Ongoing)",
    "IBM Data Science Professional Certificate (Coursera)"
  ]
}

In [33]:
js['education'][0]

'Post Graduate Program in Data Science Engineering - AI & ML, Great Lakes Institute of Management (02/2024 to 01/2025)'

In [39]:
js["passing_out_date"]["M.C.A."] 

'07-2023'

In [25]:
js

{'first_name': 'Tanya',
 'last_name': 'Dhingra',
 'email': 'Tanyadhingra186@gmail.com',
 'phone_no': '+91-8901128354',
 'education': ['Post Graduate Program in Data Science Engineering - AI & ML, Great Lakes Institute of Management (02/2024 to 01/2025)',
  'Master of Computer Applications (M.C.A.)-Software Engineering GPA: 8.97, University School of Information, Communication and Technology, GGSIPU (10/2021 to 07/2023)',
  'Bachelor of Science (B.Sc.) - Computer Science GPA: 7.95, Maitreyi College, Delhi University (07/2018 to 07/2021)'],
 'passing_out_date': {'M.C.A.': '07-2023',
  'PGP in Data Science Engineering': '01-2025',
  'B.Sc. Computer Science': '07-2021'},
 'skills': ['Tableau',
  'Power BI',
  'MySQL',
  'MongoDB',
  'IBM OpenPages',
  'MS Office Suite (Excel, PowerPoint, Access)',
  'Python',
  'R',
  'SQL',
  'Statistical Analysis',
  'Data Visualization',
  'Data Analysis'],
 'experience': ['Data Analyst, PhysicsWallah (09/2022 to 01/2025)',
  'Python Developer Intern, i

In [26]:
js['phone_no']

'+91-8901128354'

In [27]:
js['education']

['Post Graduate Program in Data Science Engineering - AI & ML, Great Lakes Institute of Management (02/2024 to 01/2025)',
 'Master of Computer Applications (M.C.A.)-Software Engineering GPA: 8.97, University School of Information, Communication and Technology, GGSIPU (10/2021 to 07/2023)',
 'Bachelor of Science (B.Sc.) - Computer Science GPA: 7.95, Maitreyi College, Delhi University (07/2018 to 07/2021)']

In [43]:
!pip install ydata-profiling streamlit
# streamlit run resume_profiler.py

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 655.7/655.7 kB 15.9 MB/s eta 0:00:00
  Created wheel for htmlmin: filename=htmlmin-0.1.12-py3-none-any.whl size=27081 sha256=bd1772ec969725574e786139f62de986e06e2bdf89a4f2992db810273fae20f1
  Stored in directory: /Users/anju.sharma/Library/Caches/pip/wheels/5f/d4/d7/4189b07b5902ee9f3ce0dbb14909fbe8037c39d6c63ffd49c9
Successfully built htmlmin


In [48]:
from pdfminer.high_level import extract_text
 
def extract_text_from_pdf(pdf_path):
    return extract_text(pdf_path)
from ollama import Client
import json
import re

def analyze_resume(text, model_name='llama3.1:8b'):
    """Analyze resume text and return structured JSON data"""
    client = Client(host='http://localhost:11434')
    
    # Structured prompt for JSON output
    prompt = f"""Extract the following information from this resume in JSON format:
    {text}
    
    Return JSON with these keys:
    - "first_name" (string)
    -"last_name"(string)
    - "email" (email)
    - "phone_no" (integer)
    -"adress (string) so that we can use this to fill in form"
    - "education" (array of strings)
    - "passing out date (date) with course mapped correctly so that we can use this to fill in form"
    - "skills" (array of strings)
    - "experience" (array of strings with job titles and durations(format dd-mm-yy) mapped correctly so that we can use this to fill in form)
    - "certifications" (array of strings, optional)
    -"main projects to map in the form (strings) with little bit description"
    
    Format: {{ "key": "value" }} without any additional text."""

    try:
        response = client.chat(model=model_name, messages=[
            {'role': 'user', 'content': prompt}
        ])
        
        # Extract JSON from response
        raw_output = response['message']['content']
        
        # Use regex to find JSON in the response
        json_match = re.search(r'\{.*\}', raw_output, re.DOTALL)
        if json_match:
            json_str = json_match.group()
            return json.loads(json_str)
        
        return {"error": "No JSON found in response"}
    
    except json.JSONDecodeError:
        print("Error decoding JSON response")
        return {"error": "Invalid JSON format"}
    except Exception as e:
        print(f"Error processing request: {str(e)}")
        return {"error": str(e)}

# Example usage
if __name__ == "__main__":

    
    print("Analyzing resume...")
    result = analyze_resume(extract_text_from_pdf('ResumeDTanyaDhingra.pdf'))
    
    print("\nStructured Resume Data:")
    print(json.dumps(result, indent=2))

Analyzing resume...

Structured Resume Data:
{
  "first_name": "Tanya",
  "last_name": "Dhingra",
  "email": "tanyadhingra186@gmail.com",
  "phone_no": "+919901128354",
  "address": "Delhi, India",
  "education": [
    {
      "course": "Post Graduate Program in Data Science Engineering - AI & ML",
      "institute": "Great Lakes Institute of Management",
      "passing_out_date": "01-2025"
    },
    {
      "course": "Master of Computer Applications (M.C.A.)-Software Engineering",
      "gpa": "8.97",
      "institute": "University School of Information, Communication and Technology, GGSIPU",
      "passing_out_date": "07-2023"
    },
    {
      "course": "Bachelor of Science (B.Sc.) - Computer Science",
      "gpa": "7.95",
      "institute": "Maitreyi College, Delhi University",
      "passing_out_date": "07-2021"
    }
  ],
  "skills": [
    "Tableau",
    "Power BI",
    "MySQL",
    "MongoDB",
    "IBM OpenPages",
    "MS Office Suite (Excel, PowerPoint, Access)",
    "Python",

In [49]:
from pdfminer.high_level import extract_text
 
def extract_text_from_pdf(pdf_path):
    return extract_text(pdf_path)
from ollama import Client
import json
import re

def analyze_resume(text, model_name='llama3.1:8b'):
    """Analyze resume text and return structured JSON data"""
    client = Client(host='http://localhost:11434')
    
    # Structured prompt for JSON output
    prompt = f"""Extract the following information from this resume in JSON format:
    {text}
    
    Return JSON with these keys:
    - "first_name" (string)
    -"last_name"(string)
    - "email" (email)
    - "phone_no" (integer)
    -"adress (string) so that we can use this to fill in form"
    - "education" (array of strings)
    - "passing out date (date) with course mapped correctly so that we can use this to fill in form"
    - "skills" (array of strings)
    - "experience" (array of strings with job titles and durations(format dd-mm-yy) mapped correctly so that we can use this to fill in form)
    - "certifications" (array of strings, optional)
    -"main projects to map in the form (strings) with little bit description"
    
    Format: {{ "key": "value" }} without any additional text."""

    try:
        response = client.chat(model=model_name, messages=[
            {'role': 'user', 'content': prompt}
        ])
        
        # Extract JSON from response
        raw_output = response['message']['content']
        
        # Use regex to find JSON in the response
        json_match = re.search(r'\{.*\}', raw_output, re.DOTALL)
        if json_match:
            json_str = json_match.group()
            return json.loads(json_str)
        
        return {"error": "No JSON found in response"}
    
    except json.JSONDecodeError:
        print("Error decoding JSON response")
        return {"error": "Invalid JSON format"}
    except Exception as e:
        print(f"Error processing request: {str(e)}")
        return {"error": str(e)}

# Example usage
if __name__ == "__main__":

    
    print("Analyzing resume...")
    result = analyze_resume(extract_text_from_pdf('ResumeDTanyaDhingra.pdf'))
    
    print("\nStructured Resume Data:")
    print(json.dumps(result, indent=2))

Analyzing resume...

Structured Resume Data:
{
  "first_name": "Tanya",
  "last_name": "Dhingra",
  "email": "tanyadhingra186@gmail.com",
  "phone_no": 8901128354,
  "address": "Delhi, India",
  "education": [
    "Post Graduate Program in Data Science Engineering - AI & ML (02/2024 to 01/2025)",
    "Master of Computer Applications (M.C.A.)-Software Engineering (10/2021 to 07/2023)",
    "Bachelor of Science (B.Sc.) - Computer Science (07/2018 to 07/2021)"
  ],
  "experience": [
    {
      "job_title": "Data Analyst",
      "duration": "09/2022 to 01/2025"
    },
    {
      "job_title": "Python Developer Intern",
      "duration": "01/2024 to 02/2024"
    },
    {
      "job_title": "Data Analyst Intern",
      "duration": "02/2021 to 04/2021"
    }
  ],
  "skills": [
    "Tableau",
    "Power BI",
    "MySQL",
    "MongoDB",
    "IBM OpenPages",
    "MS Office Suite (Excel, PowerPoint, Access)",
    "Python",
    "R",
    "SQL",
    "Statistical Analysis",
    "Data Visualization",

In [56]:
import PyPDF2
from docx import Document
import os

def extract_text_from_pdf(pdf_path):
    """
    Extract text from a PDF file.
    
    Args:
        pdf_path (str): Path to the PDF file
        
    Returns:
        str: Extracted text from the PDF
    """
    try:
        with open(pdf_path, 'rb') as file:
            # Create PDF reader object
            pdf_reader = PyPDF2.PdfReader(file)
            
            # Initialize text variable
            text = ""
            
            # Extract text from each page
            for page in pdf_reader.pages:
                text += page.extract_text() + "\n"
                
            return text.strip()
    except Exception as e:
        return f"Error extracting text from PDF: {str(e)}"

def extract_text_from_word(docx_path):
    """
    Extract text from a Word document.
    
    Args:
        docx_path (str): Path to the Word document
        
    Returns:
        str: Extracted text from the Word document
    """
    try:
        doc = Document(docx_path)
        
        # Extract text from paragraphs
        text = "\n".join([paragraph.text for paragraph in doc.paragraphs])
        return text.strip()
    except Exception as e:
        return f"Error extracting text from Word document: {str(e)}"

def extract_text_from_document(file_path):
    """
    Extract text from either PDF or Word document based on file extension.
    
    Args:
        file_path (str): Path to the document
        
    Returns:
        str: Extracted text from the document
    """
    _, file_extension = os.path.splitext(file_path)
    
    if file_extension.lower() == '.pdf':
        return extract_text_from_pdf(file_path)
    elif file_extension.lower() in ['.docx', '.doc']:
        return extract_text_from_word(file_path)
    else:
        return "Unsupported file format. Please provide a PDF or Word document."

# Example usage
if __name__ == "__main__":
    # Example with PDF
    pdf_text = extract_text_from_document("example.pdf")
    print("Text from PDF:")
    print(pdf_text)
    
    # Example with Word document
    word_text = extract_text_from_document("example.docx")
    print("\nText from Word document:")
    print(word_text)

Text from PDF:
Error extracting text from PDF: [Errno 2] No such file or directory: 'example.pdf'

Text from Word document:
Error extracting text from Word document: Package not found at 'example.docx'


In [50]:
pip install PyPDF2 python-docx

Note: you may need to restart the kernel to use updated packages.


In [54]:

def extract_text_from_word(docx_path):
    """
    Extract text from a Word document.
    
    Args:
        docx_path (str): Path to the Word document
        
    Returns:
        str: Extracted text from the Word document
    """
    try:
        doc = Document(docx_path)
        
        # Extract text from paragraphs
        text = "\n".join([paragraph.text for paragraph in doc.paragraphs])
        return text.strip()
    except Exception as e:
        return f"Error extracting text from Word document: {str(e)}"

In [57]:
import os
print(extract_text_from_document("ResumeDTanyaDhingra.docx"))

TANYA DHINGRA

Delhi, India | +91-8901128354 | Tanyadhingra186@gmail.com | WWW: https://www.linkedin.com/in/tanyadhingra13/


Working Experience
Data Analyst, PhysicsWallah	09/2022 to 01/2025
Delhi, India
Analyzed datasets with 10,000+ records, improving decision-making by 15% through actionable insights.
Developed automated dashboards in Tableau and Power BI, reducing manual reporting time by 30%.
Python Developer Intern, iTechplement	01/2024 to 02/2024
Delhi, India
Developed a command-line tool to extract, manipulate, and parse PDFs, enhancing document management efficiency.
Automated PDF extraction and data processing, reducing manual efforts by 40%.
Utilized libraries like PyPDF2 and PDFMiner to build robust functionality for text extraction, metadata retrieval, and file conversion.

Data Analyst Intern, MedTourEasy	02/2021 to 04/2021
Delhi, India
Collaborated with cross-functional teams to address data-related challenges, leading to a 10% improvement in decision-making efficiency.

In [58]:
import os
print(extract_text_from_document("ResumeDTanyaDhingra.pdf"))

Data Analyst , PhysicsWallah                                                   09/2022  to 01/2025  
          Delhi,  India  
• Analyzed datasets with 10,000+ records, improving decision -making by 15% through actionable insights.  
• Developed automated dashboards in Tableau and Power BI, reducing manual reporting time by 30%.  
 
Python  Developer  Intern , iTechplement                     01/2024 to 02/2024 
          Delhi,  India 
• Developed  a command -line tool to extract,  manipulate,  and parse  PDFs,  enhancing  document  management  efficiency.  
• Automated PDF extraction and data processing, reducing manual efforts by 40% . 
• Utilized  libraries  like PyPDF2  and PDFMiner  to build  robust  functionality  for text extraction,  metadata  retrieval,  and file conversion.  
 
Data Analyst  Intern , MedTourEasy                   02/2021 to 04/2021 
          Delhi,  India 
• Collaborated  with cross -functional  teams  to address  data-related  challenges,  leading  to a 10

In [ ]:
# Make sure to install the libraries first:
# pip install python-docx PyPDF2

import PyPDF2
from docx import Document  # Changed this line to properly import Document
import os

def extract_text_from_pdf(pdf_path):
    """
    Extract text from a PDF file.
    
    Args:
        pdf_path (str): Path to the PDF file
        
    Returns:
        str: Extracted text from the PDF
    """
    try:
        with open(pdf_path, 'rb') as file:
            # Create PDF reader object
            pdf_reader = PyPDF2.PdfReader(file)
            
            # Initialize text variable
            text = ""
            
            # Extract text from each page
            for page in pdf_reader.pages:
                text += page.extract_text() + "\n"
                
            return text.strip()
    except Exception as e:
        return f"Error extracting text from PDF: {str(e)}"

def extract_text_from_word(docx_path):
    """
    Extract text from a Word document.
    
    Args:
        docx_path (str): Path to the Word document
        
    Returns:
        str: Extracted text from the Word document
    """
    try:
        doc = Document(docx_path)
        
        # Extract text from paragraphs
        text = "\n".join([paragraph.text for paragraph in doc.paragraphs])
        return text.strip()
    except Exception as e:
        return f"Error extracting text from Word document: {str(e)}"

def extract_text_from_document(file_path):
    """
    Extract text from either PDF or Word document based on file extension.
    
    Args:
        file_path (str): Path to the document
        
    Returns:
        str: Extracted text from the document
    """
    _, file_extension = os.path.splitext(file_path)
    
    if file_extension.lower() == '.pdf':
        return extract_text_from_pdf(file_path)
    elif file_extension.lower() in ['.docx', '.doc']:
        return extract_text_from_word(file_path)
    else:
        return "Unsupported file format. Please provide a PDF or Word document."

# Example usage
if __name__ == "__main__":
    # Example with PDF
    pdf_text = extract_text_from_document("example.pdf")
    print("Text from PDF:")
    print(pdf_text)
    
    # Example with Word document
    word_text = extract_text_from_document("example.docx")
    print("\nText from Word document:")
    print(word_text)

In [62]:
from docx import Document

def extract_text_from_word(docx_path):
    """
    Extract text from a Word document.
    
    Args:
        docx_path (str): Path to the Word document
        
    Returns:
        str: Extracted text from the Word document
    """
    try:
        # Open the Word document
        doc = Document(docx_path)
        
        # Extract text from paragraphs
        text = ""
        for paragraph in doc.paragraphs:
            text += paragraph.text + "\n"
            
        return text.strip()
    except Exception as e:
        return f"Error extracting text from Word document: {str(e)}"

# Example usage
if __name__ == "__main__":
    # Replace 'example.docx' with your Word document path
    # word_text = extract_text_from_word("ResumeDTanyaDhingra.docx")
    result=analyze_resume(extract_text_from_word("ResumeDTanyaDhingra.docx"))
    print(json.dumps(result, indent=2))

{
  "first_name": "Tanya",
  "last_name": "Dhingra",
  "email": "tanyadhingra186@gmail.com",
  "phone_no": "+91-8901128354",
  "address": "Delhi, India",
  "education": [
    "Post Graduate Program in Data Science Engineering - AI & ML (02/2024 to 01/2025)",
    "Master of Computer Applications (M.C.A.)-Software Engineering GPA: 8.97 (10/2021 to 07/2023)",
    "Bachelor of Science (B.Sc.) - Computer Science GPA: 7.95 (07/2018 to 07/2021)"
  ],
  "passing_out_date": [
    {
      "course": "Post Graduate Program in Data Science Engineering - AI & ML",
      "date": "01-2025"
    },
    {
      "course": "Master of Computer Applications (M.C.A.)-Software Engineering",
      "date": "07-2023"
    },
    {
      "course": "Bachelor of Science (B.Sc.) - Computer Science",
      "date": "07-2021"
    }
  ],
  "skills": [
    "Technical Tools: Tableau, Power BI, MySQL, MongoDB, IBM OpenPages, MS Office Suite",
    "Programming Languages: Python, R, SQL",
    "Analytical Expertise: Statistical